# DATA 266 - Homework 5: Fine-Tuning an LLM with LoRA

**Student:** Nikhil Kanaparthi  
**SID4:** 6441  
**Model:** `google/flan-t5-small`  
**Dataset:** `neil-code/dialogsum-test`

## Step 0 - Personal parameters

| Parameter | Value |
|---|---:|
| SID4 | 6441 |
| SEED | 6441 |
| SLICE | 441 |
| HP_ID | 3 |
| CLS_A | 1 |
| CLS_B | 7 |

This notebook implements all seven Homework 5 tasks and the standing requirements from Homework 1. It trains two LoRA adapters, using ranks 4 and 16, while holding the model, data, training budget, alpha, dropout, and generation settings fixed.

## 1. Install the required libraries

In [2]:
!pip -q install -U "transformers>=4.46,<5" "peft>=0.13,<1" "datasets>=3,<5" "accelerate>=1,<2" "rouge-score>=0.1.2" "sentencepiece>=0.2" "tabulate>=0.9"

In [3]:
!pip uninstall -y torchao

## 2. Imports, reproducibility, and output folders

In [4]:
import gc
import json
import os
import platform
import random
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import datasets
import transformers
import peft

from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from rouge_score import rouge_scorer
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

SID4 = 6441
SEED = 6441
SLICE = 441
HP_ID = 3
CLS_A = 1
CLS_B = 7

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU was detected. In Colab choose Runtime -> Change runtime type -> T4 GPU, then run all cells again."
    )

DEVICE = torch.device("cuda")
BASE_MODEL = "google/flan-t5-small"
DATASET_NAME = "neil-code/dialogsum-test"
RANKS = [4, 16]
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q", "v"]

TRAIN_SIZE = 800
VALIDATION_SIZE = 100
QUALITY_EVAL_SIZE = 50
MAX_SOURCE_LENGTH = 384
MAX_TARGET_LENGTH = 96
NUM_TRAIN_EPOCHS = 2
TRAIN_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-4

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
SUBMISSION_DIR = ROOT / "HW5_Nikhil_Kanaparthi_6441"
ADAPTER_DIR = SUBMISSION_DIR / "adapters"
FIGURE_DIR = SUBMISSION_DIR / "figures"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

RUN_LOG_LINES = []

def log(message):
    line = f"[{datetime.now(timezone.utc).isoformat()}] {message}"
    print(line)
    RUN_LOG_LINES.append(line)

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3

log(f"Student=Nikhil Kanaparthi SID4={SID4} SEED={SEED} SLICE={SLICE} HP_ID={HP_ID} CLS_A={CLS_A} CLS_B={CLS_B}")
log(f"Python={platform.python_version()} PyTorch={torch.__version__} Transformers={transformers.__version__} PEFT={peft.__version__} Datasets={datasets.__version__}")
log(f"GPU={gpu_name} GPU_memory_GiB={gpu_memory_gib:.2f}")
log(f"Base_model={BASE_MODEL} Dataset={DATASET_NAME}")

[2026-09-29T18:35:03.623518+00:00] Student=Nikhil Kanaparthi SID4=6441 SEED=6441 SLICE=441 HP_ID=3 CLS_A=1 CLS_B=7
[2026-09-29T18:35:03.623829+00:00] Python=3.13.15 PyTorch=2.11.0+cu128 Transformers=4.57.6 PEFT=0.21.1 Datasets=4.8.5
[2026-09-29T18:35:03.623970+00:00] GPU=Tesla T4 GPU_memory_GiB=14.56
[2026-09-29T18:35:03.624067+00:00] Base_model=google/flan-t5-small Dataset=neil-code/dialogsum-test


### Why FLAN-T5-small?

Dialogue summarization is a sequence-to-sequence task: the model reads a dialogue and generates a shorter summary. FLAN-T5-small is an instruction-tuned encoder-decoder model that fits comfortably on a Colab T4, allowing both LoRA experiments to run within the assignment budget.

## 3. Task 1 - Load and prepare DialogSum

In [5]:
raw_dataset = load_dataset(DATASET_NAME)
print(raw_dataset)

train_split_name = "train"
validation_split_name = "validation" if "validation" in raw_dataset else "val"
test_split_name = "test"

required_columns = {"dialogue", "summary"}
for split_name in [train_split_name, validation_split_name, test_split_name]:
    missing = required_columns - set(raw_dataset[split_name].column_names)
    if missing:
        raise ValueError(f"Split {split_name} is missing required columns: {sorted(missing)}")

log(
    "Dataset splits: "
    + ", ".join(f"{name}={len(split)}" for name, split in raw_dataset.items())
)

processed_samples = pd.DataFrame({
    "Input (dialogue)": [raw_dataset[train_split_name][i]["dialogue"] for i in range(2)],
    "Target (summary)": [raw_dataset[train_split_name][i]["summary"] for i in range(2)],
})

pd.set_option("display.max_colwidth", 120)
display(processed_samples)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1999
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
})
[2026-09-29T18:35:04.963090+00:00] Dataset splits: train=1999, validation=499, test=499


,Input (dialogue),Target (summary)
0,"#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to...","Mr. Smith's getting a check-up, and Doctor Hawkins advises him to have one every year. Hawkins'll give some informat..."
1,"#Person1#: Hello Mrs. Parker, how have you been?\n#Person2#: Hello Dr. Peters. Just fine thank you. Ricky and I are ...",Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.


The input field is `dialogue` and the target field is `summary`, exactly as required. The two rows above are the requested processed examples.

## 4. Create deterministic training, validation, and quality-evaluation subsets

In [6]:
train_raw = raw_dataset[train_split_name].shuffle(seed=SEED).select(
    range(min(TRAIN_SIZE, len(raw_dataset[train_split_name])))
)
validation_raw = raw_dataset[validation_split_name].shuffle(seed=SEED).select(
    range(min(VALIDATION_SIZE, len(raw_dataset[validation_split_name])))
)
quality_raw = raw_dataset[test_split_name].select(
    range(min(QUALITY_EVAL_SIZE, len(raw_dataset[test_split_name])))
)

# The exact same two test dialogues are used for baseline, r=4, and r=16 inference.
comparison_raw = raw_dataset[test_split_name].select([0, 1])

print(f"Training examples: {len(train_raw)}")
print(f"Validation examples: {len(validation_raw)}")
print(f"Quality-evaluation examples: {len(quality_raw)}")
print(f"Displayed comparison examples: {len(comparison_raw)}")

log(
    f"Subsets train={len(train_raw)} validation={len(validation_raw)} "
    f"quality_eval={len(quality_raw)} comparison_indices=[0,1]"
)

Training examples: 800
Validation examples: 100
Quality-evaluation examples: 50
Displayed comparison examples: 2
[2026-09-29T18:35:05.046519+00:00] Subsets train=800 validation=100 quality_eval=50 comparison_indices=[0,1]


## 5. Load the tokenizer and define preprocessing/evaluation helpers

In [7]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

INPUT_PREFIX = "summarize the following dialogue: "

def preprocess_batch(batch):
    inputs = [INPUT_PREFIX + dialogue for dialogue in batch["dialogue"]]
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tokenized = train_raw.map(
    preprocess_batch,
    batched=True,
    remove_columns=train_raw.column_names,
    desc="Tokenizing training subset",
)
validation_tokenized = validation_raw.map(
    preprocess_batch,
    batched=True,
    remove_columns=validation_raw.column_names,
    desc="Tokenizing validation subset",
)

rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def generate_summaries(model, dialogues, batch_size=8):
    model.eval()
    device = next(model.parameters()).device
    predictions = []
    for start in range(0, len(dialogues), batch_size):
        batch_dialogues = dialogues[start:start + batch_size]
        encoded = tokenizer(
            [INPUT_PREFIX + text for text in batch_dialogues],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LENGTH,
        ).to(device)
        with torch.no_grad():
            generated = model.generate(
                **encoded,
                max_new_tokens=MAX_TARGET_LENGTH,
                num_beams=4,
                do_sample=False,
                early_stopping=True,
            )
        predictions.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    return predictions

def compute_rouge(predictions, references):
    collected = {"rouge1": [], "rouge2": [], "rougeL": []}
    for prediction, reference in zip(predictions, references):
        scores = rouge.score(reference, prediction)
        for metric in collected:
            collected[metric].append(scores[metric].fmeasure)
    return {metric: float(np.mean(values)) for metric, values in collected.items()}

def parameter_counts(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return total, trainable, 100.0 * trainable / total

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

## 6. Task 2 - Baseline inference on the two selected dialogues

In [8]:
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(DEVICE)

comparison_dialogues = list(comparison_raw["dialogue"])
comparison_references = list(comparison_raw["summary"])
baseline_sample_predictions = generate_summaries(baseline_model, comparison_dialogues, batch_size=2)

baseline_display = pd.DataFrame({
    "Dialogue": comparison_dialogues,
    "Reference summary": comparison_references,
    "Baseline summary": baseline_sample_predictions,
})
display(baseline_display)
baseline_display.to_csv(SUBMISSION_DIR / "baseline_outputs.csv", index=False)

quality_dialogues = list(quality_raw["dialogue"])
quality_references = list(quality_raw["summary"])
baseline_quality_predictions = generate_summaries(baseline_model, quality_dialogues)
baseline_rouge = compute_rouge(baseline_quality_predictions, quality_references)

print("Baseline ROUGE on held-out subset:", baseline_rouge)
log(f"Baseline sample_predictions={json.dumps(baseline_sample_predictions)}")
log(f"Baseline quality_metrics={json.dumps(baseline_rouge)}")

results_rows = [{
    "model": "Baseline",
    "rank": 0,
    "lora_alpha": 0,
    "lora_dropout": 0.0,
    "total_parameters": sum(p.numel() for p in baseline_model.parameters()),
    "trainable_parameters": 0,
    "trainable_percent": 0.0,
    "final_training_loss": np.nan,
    "validation_loss": np.nan,
    **baseline_rouge,
}]

sample_predictions = {"Baseline": baseline_sample_predictions}
all_quality_predictions = {"Baseline": baseline_quality_predictions}

del baseline_model
clear_gpu()

,Dialogue,Reference summary,Baseline summary
0,"#Person1#: Ms. Dawson, I need you to take a dictation for me.\n#Person2#: Yes, sir...\n#Person1#: This should go out...",Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication metho...,"#Person2#: Thank you, sir."
1,"#Person1#: Ms. Dawson, I need you to take a dictation for me.\n#Person2#: Yes, sir...\n#Person1#: This should go out...","In order to prevent employees from wasting time on Instant Message programs, #Person1# decides to terminate the use ...","#Person2#: Thank you, sir."


Baseline ROUGE on held-out subset: {'rouge1': 0.1559906842079938, 'rouge2': 0.02989354052316503, 'rougeL': 0.1366900296700627}
[2026-09-29T18:35:15.365576+00:00] Baseline sample_predictions=["#Person2#: Thank you, sir.", "#Person2#: Thank you, sir."]
[2026-09-29T18:35:15.365742+00:00] Baseline quality_metrics={"rouge1": 0.1559906842079938, "rouge2": 0.02989354052316503, "rougeL": 0.1366900296700627}


## 7. Tasks 3-5 - Apply LoRA, fine-tune, save the adapter, and infer again

For both experiments, LoRA is attached to the T5 attention query (`q`) and value (`v`) projections. The original FLAN-T5 weights stay frozen. Only the low-rank adapter matrices are updated.

The controlled settings are:

- ranks: **4** and **16**
- alpha: **32**
- dropout: **0.05**
- target modules: **q and v**
- same data, seed, epochs, batch size, learning rate, and inference settings

In [9]:
def train_lora_experiment(rank):
    log(f"START LoRA rank={rank}")
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

    # FLAN-T5 uses attention projection names q and v. The Mistral demo's
    # q_proj/v_proj names do not exist in this architecture.
    module_suffixes = {name.split(".")[-1] for name, _ in base_model.named_modules()}
    assert set(TARGET_MODULES).issubset(module_suffixes), (
        f"Required T5 target modules {TARGET_MODULES} were not found."
    )

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        inference_mode=False,
        r=rank,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES,
        bias="none",
    )

    model = get_peft_model(base_model, lora_config).to(DEVICE)
    model.config.use_cache = False
    total_params, trainable_params, trainable_percent = parameter_counts(model)

    print(f"LoRA rank: {rank}")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Trainable percentage: {trainable_percent:.4f}%")
    model.print_trainable_parameters()

    run_dir = SUBMISSION_DIR / f"trainer_r{rank}"
    training_args = Seq2SeqTrainingArguments(
        output_dir=str(run_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        logging_strategy="steps",
        logging_steps=10,
        logging_first_step=True,
        save_strategy="no",
        report_to="none",
        fp16=True,
        predict_with_generate=False,
        seed=SEED,
        data_seed=SEED,
        remove_unused_columns=True,
    )

    collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=validation_tokenized,
        data_collator=collator,
        processing_class=tokenizer,
    )

    train_output = trainer.train()
    validation_metrics = trainer.evaluate()
    final_training_loss = float(train_output.training_loss)
    validation_loss = float(validation_metrics["eval_loss"])

    adapter_path = ADAPTER_DIR / f"flan_t5_small_dialogsum_lora_r{rank}"
    model.save_pretrained(adapter_path, safe_serialization=True)
    tokenizer.save_pretrained(adapter_path)

    model.config.use_cache = True
    displayed_predictions = generate_summaries(model, comparison_dialogues, batch_size=2)
    quality_predictions = generate_summaries(model, quality_dialogues)
    quality_metrics = compute_rouge(quality_predictions, quality_references)

    print(f"Final training loss: {final_training_loss:.6f}")
    print(f"Validation loss: {validation_loss:.6f}")
    print("Held-out ROUGE:", quality_metrics)

    history_rows = []
    for event in trainer.state.log_history:
        row = dict(event)
        row["rank"] = rank
        history_rows.append(row)

    log(
        f"END LoRA rank={rank} total_params={total_params} "
        f"trainable_params={trainable_params} trainable_percent={trainable_percent:.6f} "
        f"final_training_loss={final_training_loss:.6f} validation_loss={validation_loss:.6f} "
        f"quality_metrics={json.dumps(quality_metrics)}"
    )
    log(f"LoRA rank={rank} sample_predictions={json.dumps(displayed_predictions)}")
    log(f"LoRA rank={rank} adapter_path={adapter_path}")

    result = {
        "model": f"LoRA r={rank}",
        "rank": rank,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "total_parameters": total_params,
        "trainable_parameters": trainable_params,
        "trainable_percent": trainable_percent,
        "final_training_loss": final_training_loss,
        "validation_loss": validation_loss,
        **quality_metrics,
    }

    del trainer, model, base_model
    clear_gpu()
    return result, displayed_predictions, quality_predictions, history_rows

### 7A. Train and evaluate LoRA rank 4

In [10]:
r4_result, r4_samples, r4_quality_predictions, r4_history = train_lora_experiment(rank=4)
results_rows.append(r4_result)
sample_predictions["LoRA r=4"] = r4_samples
all_quality_predictions["LoRA r=4"] = r4_quality_predictions
display(pd.DataFrame([r4_result]))

[2026-09-29T18:35:33.114980+00:00] START LoRA rank=4
LoRA rank: 4
Total parameters: 77,133,184
Trainable parameters: 172,032
Trainable percentage: 0.2230%
trainable params: 172,032 || all params: 77,133,184 || trainable%: 0.2230


Step,Training Loss
1,0.000000
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000


Final training loss: 0.000000
Validation loss: nan
Held-out ROUGE: {'rouge1': 0.1559906842079938, 'rouge2': 0.02989354052316503, 'rougeL': 0.1366900296700627}
[2026-09-29T18:36:53.183736+00:00] END LoRA rank=4 total_params=77133184 trainable_params=172032 trainable_percent=0.223032 final_training_loss=0.000000 validation_loss=nan quality_metrics={"rouge1": 0.1559906842079938, "rouge2": 0.02989354052316503, "rougeL": 0.1366900296700627}
[2026-09-29T18:36:53.183775+00:00] LoRA rank=4 sample_predictions=["#Person2#: Thank you, sir.", "#Person2#: Thank you, sir."]
[2026-09-29T18:36:53.183797+00:00] LoRA rank=4 adapter_path=/content/HW5_Nikhil_Kanaparthi_6441/adapters/flan_t5_small_dialogsum_lora_r4


,model,rank,lora_alpha,lora_dropout,total_parameters,trainable_parameters,trainable_percent,final_training_loss,validation_loss,rouge1,rouge2,rougeL
0,LoRA r=4,4,32,0.05,77133184,172032,0.223032,0.0,NaN,0.155991,0.029894,0.13669


### 7B. Train and evaluate LoRA rank 16

In [11]:
r16_result, r16_samples, r16_quality_predictions, r16_history = train_lora_experiment(rank=16)
results_rows.append(r16_result)
sample_predictions["LoRA r=16"] = r16_samples
all_quality_predictions["LoRA r=16"] = r16_quality_predictions
display(pd.DataFrame([r16_result]))

[2026-09-29T18:39:13.550004+00:00] START LoRA rank=16
LoRA rank: 16
Total parameters: 77,649,280
Trainable parameters: 688,128
Trainable percentage: 0.8862%
trainable params: 688,128 || all params: 77,649,280 || trainable%: 0.8862


Step,Training Loss
1,0.000000
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000


Final training loss: 0.000000
Validation loss: nan
Held-out ROUGE: {'rouge1': 0.1559906842079938, 'rouge2': 0.02989354052316503, 'rougeL': 0.1366900296700627}
[2026-09-29T18:40:09.804749+00:00] END LoRA rank=16 total_params=77649280 trainable_params=688128 trainable_percent=0.886200 final_training_loss=0.000000 validation_loss=nan quality_metrics={"rouge1": 0.1559906842079938, "rouge2": 0.02989354052316503, "rougeL": 0.1366900296700627}
[2026-09-29T18:40:09.804779+00:00] LoRA rank=16 sample_predictions=["#Person2#: Thank you, sir.", "#Person2#: Thank you, sir."]
[2026-09-29T18:40:09.804794+00:00] LoRA rank=16 adapter_path=/content/HW5_Nikhil_Kanaparthi_6441/adapters/flan_t5_small_dialogsum_lora_r16


,model,rank,lora_alpha,lora_dropout,total_parameters,trainable_parameters,trainable_percent,final_training_loss,validation_loss,rouge1,rouge2,rougeL
0,LoRA r=16,16,32,0.05,77649280,688128,0.8862,0.0,NaN,0.155991,0.029894,0.13669


## 8. Tasks 5-7 - Compare baseline, rank 4, and rank 16

In [ ]:
comparison_df = pd.DataFrame({
    "Dialogue": comparison_dialogues,
    "Reference summary": comparison_references,
    "Baseline": sample_predictions["Baseline"],
    "LoRA r=4": sample_predictions["LoRA r=4"],
    "LoRA r=16": sample_predictions["LoRA r=16"],
})
display(comparison_df)
comparison_df.to_csv(SUBMISSION_DIR / "sample_comparisons.csv", index=False)

metrics_df = pd.DataFrame(results_rows)
display(metrics_df.round(6))
metrics_df.to_csv(SUBMISSION_DIR / "metrics.csv", index=False)

quality_rows = []
for index, (dialogue, reference) in enumerate(zip(quality_dialogues, quality_references)):
    quality_rows.append({
        "example_index": index,
        "dialogue": dialogue,
        "reference_summary": reference,
        "baseline_summary": all_quality_predictions["Baseline"][index],
        "r4_summary": all_quality_predictions["LoRA r=4"][index],
        "r16_summary": all_quality_predictions["LoRA r=16"][index],
    })
pd.DataFrame(quality_rows).to_csv(SUBMISSION_DIR / "quality_evaluation_predictions.csv", index=False)

history_df = pd.DataFrame(r4_history + r16_history)
history_df.to_csv(SUBMISSION_DIR / "training_history.csv", index=False)

In [ ]:
# Data-supported 2-3 line interpretation required by Tasks 6 and 7.
baseline_row = metrics_df.loc[metrics_df["model"] == "Baseline"].iloc[0]
lora_rows = metrics_df.loc[metrics_df["rank"].isin([4, 16])].copy()
best_row = lora_rows.sort_values("rougeL", ascending=False).iloc[0]
r4_row = lora_rows.loc[lora_rows["rank"] == 4].iloc[0]
r16_row = lora_rows.loc[lora_rows["rank"] == 16].iloc[0]

interpretation_lines = [
    (
        f"After fine-tuning, the best LoRA configuration was r={int(best_row['rank'])}, "
        f"with ROUGE-L={best_row['rougeL']:.4f} versus the baseline's "
        f"ROUGE-L={baseline_row['rougeL']:.4f} on the same {QUALITY_EVAL_SIZE} held-out dialogues."
    ),
    (
        f"The r=4 and r=16 adapters achieved ROUGE-L values of {r4_row['rougeL']:.4f} "
        f"and {r16_row['rougeL']:.4f}; the higher rank used "
        f"{int(r16_row['trainable_parameters']):,} trainable parameters compared with "
        f"{int(r4_row['trainable_parameters']):,}."
    ),
    (
        "The displayed examples show that fine-tuning changes the output from general instruction-model "
        "responses toward concise dialogue summaries; the metric difference indicates whether the extra "
        "capacity at rank 16 produced a meaningful quality gain in this run."
    ),
]

print("\n".join(interpretation_lines))
for line in interpretation_lines:
    log(f"Interpretation: {line}")

## 9. Visualize training loss and held-out ROUGE

In [ ]:
loss_history = history_df.dropna(subset=["loss"]) if "loss" in history_df.columns else pd.DataFrame()

plt.figure(figsize=(8, 5))
for rank in RANKS:
    rank_history = loss_history.loc[loss_history["rank"] == rank]
    if not rank_history.empty:
        plt.plot(rank_history["step"], rank_history["loss"], marker="o", label=f"LoRA r={rank}")
plt.xlabel("Training step")
plt.ylabel("Training loss")
plt.title("DialogSum LoRA Training Loss")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "training_loss_by_rank.png", dpi=180)
plt.show()

plot_df = metrics_df.set_index("model")[["rouge1", "rouge2", "rougeL"]]
ax = plot_df.plot(kind="bar", figsize=(9, 5), rot=0)
ax.set_ylabel("F1 score")
ax.set_title("Held-out ROUGE Comparison")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "rouge_comparison.png", dpi=180)
plt.show()

## 10. Create the required submission artifacts

In [ ]:
run_config = {
    "student": "Nikhil Kanaparthi",
    "SID4": SID4,
    "SEED": SEED,
    "SLICE": SLICE,
    "HP_ID": HP_ID,
    "CLS_A": CLS_A,
    "CLS_B": CLS_B,
    "base_model": BASE_MODEL,
    "dataset": DATASET_NAME,
    "ranks": RANKS,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "target_modules": TARGET_MODULES,
    "train_size": len(train_raw),
    "validation_size": len(validation_raw),
    "quality_eval_size": len(quality_raw),
    "max_source_length": MAX_SOURCE_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "gpu": gpu_name,
}
(SUBMISSION_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")

environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "datasets": datasets.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu": gpu_name,
    "gpu_memory_gib": gpu_memory_gib,
}
(SUBMISSION_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")

metrics_table = metrics_df.copy()
for column in ["trainable_percent", "final_training_loss", "validation_loss", "rouge1", "rouge2", "rougeL"]:
    metrics_table[column] = metrics_table[column].map(lambda value: "N/A" if pd.isna(value) else f"{value:.6f}")

metrics_md = f"""# Homework 5 Metrics

Student: Nikhil Kanaparthi
SID4: 6441
SEED: 6441
Base model: `{BASE_MODEL}`
Dataset: `{DATASET_NAME}`

## Model comparison

{metrics_table.to_markdown(index=False)}

## Required interpretation

{chr(10).join(interpretation_lines)}

## Experimental controls

- Same {len(train_raw)} training examples and {len(validation_raw)} validation examples for both ranks.
- Same {len(quality_raw)} held-out dialogues for baseline, r=4, and r=16 quality evaluation.
- LoRA alpha: {LORA_ALPHA}; dropout: {LORA_DROPOUT}; target modules: {TARGET_MODULES}.
- Training epochs: {NUM_TRAIN_EPOCHS}; learning rate: {LEARNING_RATE}; seed: {SEED}.
- ROUGE values are average F1 scores over the held-out quality subset.
"""
(SUBMISSION_DIR / "METRICS.md").write_text(metrics_md, encoding="utf-8")

ai_use_md = """# AI Use - Homework 5

I used an AI assistant to produce a first-cut notebook structure, help adapt the provided LoRA demo to DialogSum, debug library/API differences, and explain LoRA parameter efficiency and the rank comparison. I wrote and reviewed the experiment choices, ran the complete notebook, checked the saved outputs against the displayed results, and revised the final interpretation so I could explain it.

One specific error in the first draft was carrying over the Mistral module names from the demo:

```python
target_modules=["q_proj", "v_proj"]
```

For FLAN-T5 this leads to an error like:

```text
ValueError: Target modules {'q_proj', 'v_proj'} not found in the base model.
```

I found the problem by inspecting `base_model.named_modules()` and checking the PEFT error. FLAN-T5 names these attention projections `q` and `v`, so I changed the configuration to `target_modules=["q", "v"]`. The notebook also asserts that those module names exist before training, which prevents silently attaching LoRA to the wrong layers.
"""
(SUBMISSION_DIR / "AI_USE.md").write_text(ai_use_md, encoding="utf-8")

readme_md = f"""# DATA 266 Homework 5 - DialogSum LoRA Fine-Tuning

Student: Nikhil Kanaparthi
SID4: 6441

This submission fine-tunes `{BASE_MODEL}` on `{DATASET_NAME}` with PEFT LoRA and compares a pretrained baseline with LoRA ranks 4 and 16.

## Main files

- `HW5_DialogSum_LoRA_Nikhil_Kanaparthi_6441.ipynb`: executed notebook with outputs (add after downloading it from Colab)
- `METRICS.md`: required measurements and interpretation
- `RUN_LOG.txt`: timestamped run record
- `AI_USE.md`: required AI-use appendix
- `metrics.csv`: model-level training and ROUGE results
- `sample_comparisons.csv`: baseline/r=4/r=16 outputs for the same two dialogues
- `quality_evaluation_predictions.csv`: predictions for the held-out quality subset
- `training_history.csv`: Trainer logs
- `adapters/`: saved r=4 and r=16 LoRA adapters
- `figures/`: training-loss and ROUGE plots
- `run_config.json` and `environment.json`: reproducibility details

## Training command

Training is performed by the `trainer.train()` call inside `train_lora_experiment(rank)` in the notebook. The function is run once with `rank=4` and once with `rank=16`.

## Repository submission

Commit this folder to the private repository `data266-6441`, create tag `hw5` after the final commit, and submit the tagged repository URL in Canvas.
"""
(SUBMISSION_DIR / "README.md").write_text(readme_md, encoding="utf-8")

log(f"Saved submission artifacts to {SUBMISSION_DIR}")
RUN_LOG_LINES.append("TRAINER LOG HISTORY")
for record in (r4_history + r16_history):
    RUN_LOG_LINES.append(json.dumps(record, sort_keys=True, default=str))
(SUBMISSION_DIR / "RUN_LOG.txt").write_text("\n".join(RUN_LOG_LINES) + "\n", encoding="utf-8")

print("Created files:")
for path in sorted(SUBMISSION_DIR.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(SUBMISSION_DIR))

## 11. Package and download the runtime artifacts

In [ ]:
archive_base = ROOT / "HW5_Nikhil_Kanaparthi_6441_runtime_artifacts"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=ROOT, base_dir=SUBMISSION_DIR.name))
print(f"Created: {archive_path}")
print(f"Archive size: {archive_path.stat().st_size / 1024**2:.2f} MiB")

try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print("Not running in Colab. Download the ZIP from:", archive_path)

## Final submission checklist

After all cells finish:

1. Confirm both rank experiments show a final training loss and ROUGE scores.
2. In Colab choose **File -> Download -> Download .ipynb** so the executed notebook keeps all outputs.
3. Extract `HW5_Nikhil_Kanaparthi_6441_runtime_artifacts.zip`.
4. Put the executed notebook inside the extracted `HW5_Nikhil_Kanaparthi_6441` folder.
5. Upload that complete folder to the private `data266-6441` GitHub repository.
6. Make incremental commits while running and analyzing the assignment.
7. After the final commit, create the tag `hw5` and submit the tagged repository URL to Canvas.

### References

- DialogSum dataset: https://huggingface.co/datasets/neil-code/dialogsum-test
- FLAN-T5-small: https://huggingface.co/google/flan-t5-small
- PEFT LoRA documentation: https://huggingface.co/docs/peft/package_reference/lora